# 00 · Core: shared code and its tests

Every part notebook starts with `%run ./00_core.ipynb`. Data loading, ground truth, recall@10, the timing harness, index building, the results writer and the figure and table code therefore exist exactly once, and every part measures the same way. Each feature cell is followed by its pytest cell, run with `ipytest`; a failing test raises and stops `run_all.sh`.

The mode comes from `ANN_MODE`: `full`, or `quick` (the default, for interactive use). `run_all.sh` sets it. Measurement choices are explained in `design_note.md`.

In [ ]:
import contextlib
import dataclasses
import hashlib
import json
import os
import platform
import re
import subprocess
import time
from pathlib import Path

import faiss
import h5py
import ipytest
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytest
import threadpoolctl
from matplotlib import ticker

ipytest.autoconfig(raise_on_error=True)

## Configuration
Every size, grid and query range for both modes lives here, so `--quick` runs exactly the same code at a smaller scale.

In [ ]:
def find_root():
    if "ANN_ROOT" in os.environ:
        return Path(os.environ["ANN_ROOT"]).resolve()
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "run_all.sh").exists():
            return path
    raise FileNotFoundError("no run_all.sh in the current directory or above it")


MODE = os.environ.get("ANN_MODE", "quick")
assert MODE in ("full", "quick"), f"ANN_MODE must be 'full' or 'quick', not {MODE!r}"
ROOT = find_root()
DATA_DIR = ROOT / "data"
CACHE_DIR = ROOT / "cache"
RESULTS_DIR = ROOT / ("results" if MODE == "full" else "results_quick")
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = ROOT / "report" / "tables" if MODE == "full" else RESULTS_DIR / "tables"
for directory in (CACHE_DIR, FIGURES_DIR, TABLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
K = 10

_SHARED = dict(
    repeats=3,           # timed passes per setting; the median is reported
    warmup=200,          # untimed queries before each timed setting
    recall_stop=0.995,   # a sweep stops once tune-half recall reaches this
    gt_k=100,            # depth of the cached ground truth
    hnsw_efC=200,
    scaling_M=16,
    noise_M=16,
    noise_sigmas=[0.0, 0.1, 0.3, 1.0],  # 0.0 is the unchanged control
    noise_ef_search=[10, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024, 1536, 2048],
)
FULL = dict(
    _SHARED,
    n_base=None,                        # every base vector
    tune=(0, 5_000),
    report=(5_000, 10_000),
    exact_timing=(5_000, 6_000),        # report queries used to time exact search one at a time
    agreement_queries=10_000,           # queries checked against the stored neighbours (Part 1)
    hnsw_M=[8, 16, 32],
    ef_search=[10, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768],
    ivf_nlist=[1024, 4096],
    nprobe=[1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256],
    ivfpq_nlist=4096,
    ivfpq_m={"sift": [16, 32, 64], "glove": [20, 50]},
    scaling_sizes=[100_000, 300_000, 1_000_000],
    noise_n_base=150_000,               # 3 noisy settings x (150K base + 10K queries) = 480K generated vectors
    noise_nlist=1024,
    noise_nprobe=[1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024],
)
QUICK = dict(
    _SHARED,
    n_base=100_000,                     # the first 100,000 base vectors
    tune=(0, 1_000),
    report=(5_000, 6_000),
    exact_timing=(5_000, 6_000),
    agreement_queries=1_000,
    hnsw_M=[16],
    ef_search=[10, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512],
    ivf_nlist=[256],
    nprobe=[1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128],
    ivfpq_nlist=256,
    ivfpq_m={"sift": [32], "glove": [20]},
    scaling_sizes=[10_000, 30_000, 100_000],
    noise_n_base=50_000,
    noise_nlist=256,
    noise_nprobe=[1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256],
)
CONFIG = FULL if MODE == "full" else QUICK

In [ ]:
%%ipytest -qq

@pytest.mark.parametrize("config", [FULL, QUICK], ids=["full", "quick"])
def test_query_ranges_follow_the_assignment_split(config):
    (t0, t1), (r0, r1), (e0, e1) = config["tune"], config["report"], config["exact_timing"]
    assert 0 <= t0 < t1 <= 5_000          # tune only on the first 5,000 queries
    assert 5_000 <= r0 < r1 <= 10_000     # report only on the last 5,000
    assert r0 <= e0 < e1 <= r1            # exact timing uses report queries


@pytest.mark.parametrize("config", [FULL, QUICK], ids=["full", "quick"])
def test_pq_sub_vectors_divide_the_dimension(config):
    for name, dim in (("sift", 128), ("glove", 100)):
        assert all(dim % m == 0 for m in config["ivfpq_m"][name])


@pytest.mark.parametrize("config", [FULL, QUICK], ids=["full", "quick"])
def test_grids_increase_and_part4_follows_its_rules(config):
    for key in ("ef_search", "nprobe", "scaling_sizes", "noise_sigmas", "noise_ef_search", "noise_nprobe"):
        assert config[key] == sorted(config[key])
    assert config["noise_sigmas"][0] == 0.0 and len(config["noise_sigmas"]) >= 4  # control + 3 settings
    generated = (len(config["noise_sigmas"]) - 1) * (config["noise_n_base"] + 10_000)  # noisy base + noisy queries
    assert generated <= 500_000                                                     # at most 500K synthetic vectors
    assert config["noise_nprobe"][-1] == config["noise_nlist"]                      # IVF can always scan every list

## Machine record
Stored in every results row, so each number carries the machine and library versions it came from. The timing thread is pinned to one performance core (CPU 2 here; see `design_note.md`).

In [ ]:
def parse_cpu_list(text):
    """'0-3,8,10-11' -> [0, 1, 2, 3, 8, 10, 11]"""
    cpus = []
    for part in text.strip().split(","):
        if part:
            low, _, high = part.partition("-")
            cpus.extend(range(int(low), int(high or low) + 1))
    return cpus


def _read(path, default=""):
    try:
        return Path(path).read_text().strip()
    except OSError:
        return default


def performance_cores():
    """CPUs of the performance cores on a hybrid Intel CPU; [] where the kernel doesn't report them."""
    return parse_cpu_list(_read("/sys/devices/cpu_core/cpus"))


def default_timing_cpu():
    allowed = sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else []
    candidates = [cpu for cpu in performance_cores() if cpu in allowed] or allowed
    # The third CPU in the list (CPU 2 here): a P-core thread, away from CPU 0, which takes more interrupts.
    return candidates[min(2, len(candidates) - 1)] if candidates else None


TIMING_CPU = int(os.environ["ANN_TIMING_CPU"]) if "ANN_TIMING_CPU" in os.environ else default_timing_cpu()
BUILD_THREADS = len(performance_cores()) or os.cpu_count()


def blas_libraries():
    """BLAS libraries loaded in this process: FAISS bundles its own OpenBLAS, numpy has another."""
    return "; ".join(f"{lib.get('internal_api')} {lib.get('version')} {lib.get('architecture') or ''} "
                     f"({Path(lib['filepath']).name})".replace("  ", " ")
                     for lib in threadpoolctl.threadpool_info() if lib.get("user_api") == "blas")


def machine_info():
    cpuinfo = _read("/proc/cpuinfo").splitlines()
    meminfo = _read("/proc/meminfo").splitlines()
    cpu_model = next((line.split(":", 1)[1].strip() for line in cpuinfo if line.startswith("model name")), platform.processor())
    mem_kb = next((int(line.split()[1]) for line in meminfo if line.startswith("MemTotal")), 0)
    return dict(
        cpu_model=cpu_model,
        logical_cpus=os.cpu_count(),
        ram_gb=round(mem_kb / 2**20, 1),
        os=platform.platform(),
        python=platform.python_version(),
        numpy=np.__version__,
        faiss=faiss.__version__,
        h5py=h5py.__version__,
        pandas=pd.__version__,
        matplotlib=matplotlib.__version__,
        blas=blas_libraries(),
        governor=_read(f"/sys/devices/system/cpu/cpu{TIMING_CPU or 0}/cpufreq/scaling_governor", "unknown"),
        timing_cpu=TIMING_CPU,
        build_threads=BUILD_THREADS,
    )


MACHINE = machine_info()
MACHINE

In [ ]:
%%ipytest -qq

def test_library_versions_and_blas_are_recorded():
    assert all(MACHINE[key] for key in ("numpy", "faiss", "h5py", "pandas", "matplotlib"))
    assert "openblas" in MACHINE["blas"].lower()


def test_parse_cpu_list():
    assert parse_cpu_list("0-3,8,10-11") == [0, 1, 2, 3, 8, 10, 11]
    assert parse_cpu_list("5") == [5]
    assert parse_cpu_list("") == []


def test_machine_record_is_complete():
    assert set(MACHINE) >= {"cpu_model", "ram_gb", "os", "python", "numpy", "faiss", "governor", "timing_cpu"}
    assert MACHINE["ram_gb"] > 0 and BUILD_THREADS >= 1


@pytest.mark.skipif(not hasattr(os, "sched_getaffinity"), reason="no CPU affinity on this platform")
def test_timing_cpu_is_one_we_may_run_on():
    assert TIMING_CPU in os.sched_getaffinity(0)

## Data
SIFT and GloVe come from the ann-benchmarks HDF5 files. GloVe base vectors and queries are scaled to unit length, so Euclidean distance ranks them exactly as angular distance does. The stored neighbours are kept only when the full base set is loaded, because they are wrong for any subset. Queries 0–4,999 are for tuning and 5,000–9,999 for reported numbers (quick mode uses the first 1,000 of each).

In [ ]:
DATASETS = {
    "sift": dict(file="sift-128-euclidean.hdf5", normalise=False),
    "glove": dict(file="glove-100-angular.hdf5", normalise=True),
}


@dataclasses.dataclass
class Dataset:
    name: str
    base: np.ndarray                     # (n, d) float32, C-contiguous
    queries: np.ndarray                  # (10_000, d) float32
    stored_neighbors: np.ndarray | None  # HDF5 ground truth; None unless the full base set is loaded
    stored_distances: np.ndarray | None

    @property
    def dim(self):
        return self.base.shape[1]


def normalise(x):
    """Scale rows to unit L2 norm; all-zero rows stay zero instead of becoming NaN."""
    x = np.asarray(x, dtype=np.float32)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return np.divide(x, norms, out=np.zeros_like(x), where=norms > 0)


def load_dataset(name, n_base=None, data_dir=None):
    """Base vectors (the first n_base, or all of them) and the 10,000 queries, as float32."""
    spec = DATASETS[name]
    with h5py.File(Path(data_dir or DATA_DIR) / spec["file"], "r") as f:
        full = n_base is None or n_base >= f["train"].shape[0]
        base = f["train"][:] if full else f["train"][:n_base]
        queries = f["test"][:]
        neighbors = f["neighbors"][:] if full else None
        distances = f["distances"][:] if full else None
    base = np.ascontiguousarray(base, dtype=np.float32)
    queries = np.ascontiguousarray(queries, dtype=np.float32)
    if spec["normalise"]:
        base, queries = normalise(base), normalise(queries)
    return Dataset(name, base, queries, neighbors, distances)


TUNE = np.arange(*CONFIG["tune"])
REPORT = np.arange(*CONFIG["report"])

In [ ]:
%%ipytest -qq

def write_toy_hdf5(path, n=50, d=4, nq=10):
    rng = np.random.default_rng(1)
    with h5py.File(path, "w") as f:
        f["train"] = rng.random((n, d))
        f["test"] = rng.random((nq, d))
        f["neighbors"] = np.zeros((nq, 5), dtype=np.int32)
        f["distances"] = np.zeros((nq, 5))


def test_normalise_gives_unit_rows_and_keeps_zero_rows():
    y = normalise(np.array([[3, 4], [0, 0], [1e-3, 0]], dtype=np.float32))
    assert y.dtype == np.float32 and not np.isnan(y).any()
    assert np.allclose(np.linalg.norm(y[[0, 2]], axis=1), 1)
    assert (y[1] == 0).all()


def test_load_dataset_subsets_and_normalises(tmp_path):
    write_toy_hdf5(tmp_path / "glove-100-angular.hdf5")
    full = load_dataset("glove", data_dir=tmp_path)
    part = load_dataset("glove", n_base=20, data_dir=tmp_path)
    assert full.base.dtype == np.float32 and full.base.flags.c_contiguous
    assert np.allclose(np.linalg.norm(full.base, axis=1), 1)
    assert part.base.shape == (20, 4) and (part.base == full.base[:20]).all()
    assert full.stored_neighbors is not None and part.stored_neighbors is None


def test_query_halves_are_disjoint_and_on_the_right_side():
    assert set(TUNE).isdisjoint(REPORT)
    assert TUNE.max() < 5_000 <= REPORT.min() and REPORT.max() < 10_000

## Ground truth
This is our own exact search: a batched `IndexFlatL2` run with all build threads. It is never timed, and it is cached in `cache/`. The cache file name includes a fingerprint of the data, so a different subset or noise level never reuses the wrong file.

In [ ]:
@contextlib.contextmanager
def faiss_threads(n):
    old = faiss.omp_get_max_threads()
    faiss.omp_set_num_threads(n)
    try:
        yield
    finally:
        faiss.omp_set_num_threads(old)


def exact_knn(base, queries, k):
    """Exact k nearest neighbours by squared L2 distance (batched, multi-threaded, untimed)."""
    index = faiss.IndexFlatL2(base.shape[1])
    index.add(base)
    with faiss_threads(BUILD_THREADS):
        return index.search(queries, k)


def fingerprint(*arrays):
    digest = hashlib.sha1()
    for a in arrays:
        digest.update(str(a.shape).encode())
        digest.update(np.ascontiguousarray(a[:: max(1, len(a) // 1000)]).tobytes())
    return digest.hexdigest()[:12]


def ground_truth(tag, base, queries, k=None, cache_dir=None):
    """Exact top-k ids for every query, cached under a name tied to the data itself."""
    k = k or CONFIG["gt_k"]
    path = Path(cache_dir or CACHE_DIR) / f"gt_{tag}_{len(base)}x{base.shape[1]}_q{len(queries)}_k{k}_{fingerprint(base, queries)}.npy"
    if path.exists():
        return np.load(path)
    ids = exact_knn(base, queries, k)[1]
    np.save(path, ids)
    return ids

In [ ]:
%%ipytest -qq

def random_data(n=300, d=8, nq=20, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n, d), dtype=np.float32), rng.random((nq, d), dtype=np.float32)


def test_exact_knn_matches_numpy_brute_force():
    base, queries = random_data()
    distances, ids = exact_knn(base, queries, 10)
    true = ((queries[:, None, :].astype(np.float64) - base[None]) ** 2).sum(-1)
    assert (ids == np.argsort(true, axis=1)[:, :10]).all()
    assert np.allclose(distances, np.sort(true, axis=1)[:, :10], rtol=1e-4)


def test_ground_truth_is_cached_and_keyed_by_the_data(tmp_path):
    base, queries = random_data()
    first = ground_truth("toy", base, queries, k=5, cache_dir=tmp_path)
    assert len(list(tmp_path.glob("gt_toy_*.npy"))) == 1
    assert (ground_truth("toy", base, queries, k=5, cache_dir=tmp_path) == first).all()
    ground_truth("toy", base + 1, queries, k=5, cache_dir=tmp_path)  # different data, so a new file
    assert len(list(tmp_path.glob("gt_toy_*.npy"))) == 2

## Recall@10
A returned id counts as correct if its distance to the query is at most the distance of the true 10th neighbour. Ties at rank 10 are therefore not mistakes. Both distances are recomputed in float64 from the stored vectors, with a small relative tolerance. A FAISS `-1` (no result) is a miss; used as a numpy index it would silently pick the last vector. An id returned twice counts once.

In [ ]:
def recall_at_k(found, base, queries, gt_ids, k=K, rtol=1e-6, per_query=False):
    """Tie-aware recall@k: the share of returned ids that are no farther than the true k-th neighbour."""
    found = np.asarray(found)[:, :k]
    q = queries.astype(np.float64)
    kth = base[gt_ids[:, k - 1]].astype(np.float64)
    threshold = ((kth - q) ** 2).sum(axis=1) * (1 + rtol)

    valid = found >= 0
    vectors = base[np.where(valid, found, 0)].astype(np.float64)
    distance = ((vectors - q[:, None, :]) ** 2).sum(axis=2)

    # Flag every repeat of an id after its first appearance in the row, so duplicates count once.
    order = np.argsort(found, axis=1, kind="stable")
    ids_sorted = np.take_along_axis(found, order, axis=1)
    repeat_sorted = np.zeros_like(valid)
    repeat_sorted[:, 1:] = ids_sorted[:, 1:] == ids_sorted[:, :-1]
    repeat = np.zeros_like(valid)
    np.put_along_axis(repeat, order, repeat_sorted, axis=1)

    correct = valid & ~repeat & (distance <= threshold[:, None])
    scores = correct.sum(axis=1) / k
    return scores if per_query else float(scores.mean())

In [ ]:
%%ipytest -qq

def line_data():
    """A query at the origin and 20 points on the x-axis at distances 1..20: the true top 10 is ids 0..9."""
    base = np.zeros((20, 2), dtype=np.float32)
    base[:, 0] = np.arange(1, 21)
    return base, np.zeros((1, 2), dtype=np.float32), np.arange(20)[None, :]


def with_point(base, point):
    return np.vstack([base, [point]]).astype(np.float32)


def test_perfect_result_scores_one():
    base, q, gt = line_data()
    assert recall_at_k(gt[:, :10], base, q, gt) == 1.0


def test_wrong_result_scores_zero():
    base, q, gt = line_data()
    assert recall_at_k(np.arange(10, 20)[None, :], base, q, gt) == 0.0


def test_partial_result():
    base, q, gt = line_data()
    assert recall_at_k(np.array([[0, 1, 2, 3, 4, 5, 6, 15, 16, 17]]), base, q, gt) == pytest.approx(0.7)


def test_tie_at_the_tenth_neighbour_is_not_a_mistake():
    base, q, gt = line_data()
    base = with_point(base, [0, 10])  # id 20 is exactly as far away as the 10th neighbour (id 9)
    assert recall_at_k(np.array([[0, 1, 2, 3, 4, 5, 6, 7, 8, 20]]), base, q, gt) == 1.0


def test_point_just_beyond_the_tenth_neighbour_is_a_mistake():
    base, q, gt = line_data()
    base = with_point(base, [0, 10.01])
    assert recall_at_k(np.array([[0, 1, 2, 3, 4, 5, 6, 7, 8, 20]]), base, q, gt) == pytest.approx(0.9)


def test_minus_one_is_a_miss_not_the_last_vector():
    base, q, _ = line_data()
    base = with_point(base, [0.5, 0])        # the last vector (id 20) is now the nearest neighbour
    gt = np.array([[20, *range(19)]])
    found = np.array([[0, 1, 2, 3, 4, 5, 6, 7, -1, -1]])
    assert recall_at_k(found, base, q, gt) == pytest.approx(0.8)


def test_duplicate_ids_count_once():
    base, q, gt = line_data()
    assert recall_at_k(np.zeros((1, 10), dtype=np.int64), base, q, gt) == pytest.approx(0.1)


def test_order_does_not_matter():
    base, q, gt = line_data()
    found = np.array([[9, 3, 15, 0, 7, 12, 1, 5, 2, 8]])
    assert recall_at_k(found, base, q, gt) == recall_at_k(np.sort(found), base, q, gt) == pytest.approx(0.8)


def test_mean_and_per_query_scores():
    base, q, gt = line_data()
    found = np.vstack([gt[:, :10], np.arange(10, 20)[None, :]])
    queries, gts = np.vstack([q, q]), np.vstack([gt, gt])
    assert recall_at_k(found, base, queries, gts) == pytest.approx(0.5)
    assert recall_at_k(found, base, queries, gts, per_query=True).tolist() == [1.0, 0.0]

## Timing
QPS is measured one query at a time on a single thread: one FAISS thread, one BLAS thread, and the calling thread pinned to `TIMING_CPU`. Queries are sliced into one-row float32 views before the clock starts. Each setting gets untimed warm-up queries, then several timed passes, and the median is reported. FAISS counts the distances its searches compute whether or not anyone reads the counters; they are reset after the warm-up and read after the timed passes, so counting adds no work to the timed loop. Batch timing (Part 1 only) runs either the same pinned single thread or `n` unpinned threads.

In [ ]:
@contextlib.contextmanager
def single_thread_pinned(cpu=None):
    cpu = TIMING_CPU if cpu is None else cpu
    can_pin = hasattr(os, "sched_setaffinity") and cpu is not None
    old_affinity = os.sched_getaffinity(0) if can_pin else None
    with faiss_threads(1), threadpoolctl.threadpool_limits(1):
        if can_pin:
            os.sched_setaffinity(0, {cpu})
        try:
            yield
        finally:
            if can_pin:
                os.sched_setaffinity(0, old_affinity)


@contextlib.contextmanager
def search_threads(n):
    """n == 1: the pinned single-thread setup used for every QPS number; n > 1: n FAISS/BLAS threads, unpinned."""
    if n == 1:
        with single_thread_pinned():
            yield
    else:
        with faiss_threads(n), threadpoolctl.threadpool_limits(n):
            yield


def reset_distance_counters():
    faiss.cvar.hnsw_stats.reset()
    faiss.cvar.indexIVF_stats.reset()


def counted_distances():
    """Distances FAISS has counted since the last reset (HNSW searches, and entries scanned in IVF lists)."""
    return faiss.cvar.hnsw_stats.ndis + faiss.cvar.indexIVF_stats.ndis


def time_one_at_a_time(index, queries, k=K, repeats=None, warmup=None):
    """Median QPS of index.search called once per query, every pass's time, the ids from the last pass, and the
    distances FAISS counted per search during the timed passes."""
    repeats = repeats or CONFIG["repeats"]
    warmup = CONFIG["warmup"] if warmup is None else warmup
    queries = np.ascontiguousarray(queries, dtype=np.float32)
    rows = [queries[i : i + 1] for i in range(len(queries))]
    ids = np.empty((len(rows), k), dtype=np.int64)
    id_rows = [ids[i : i + 1] for i in range(len(rows))]
    distances = np.empty((1, k), dtype=np.float32)
    times = []
    with search_threads(1):
        for i in range(min(warmup, len(rows))):
            index.search(rows[i], k, D=distances, I=id_rows[i])
        reset_distance_counters()
        for _ in range(repeats):
            start = time.perf_counter()
            for row, id_row in zip(rows, id_rows):
                index.search(row, k, D=distances, I=id_row)
            times.append(time.perf_counter() - start)
    return dict(qps=len(rows) / float(np.median(times)), times=times, ids=ids,
                counted_per_search=counted_distances() / (repeats * len(rows)))


def time_batch(index, queries, k=K, threads=1, repeats=None):
    """Median QPS of a single search call over all queries with `threads` threads."""
    repeats = repeats or CONFIG["repeats"]
    queries = np.ascontiguousarray(queries, dtype=np.float32)
    times = []
    with search_threads(threads):
        index.search(queries[: CONFIG["warmup"]], k)
        for _ in range(repeats):
            start = time.perf_counter()
            index.search(queries, k)
            times.append(time.perf_counter() - start)
    return dict(qps=len(queries) / float(np.median(times)), times=times)

In [ ]:
%%ipytest -qq

class CountingIndex:
    """Stands in for a FAISS index: counts search calls and returns ids 0..k-1."""

    def __init__(self):
        self.calls = 0

    def search(self, x, k, D=None, I=None):
        self.calls += 1
        I = np.empty((len(x), k), dtype=np.int64) if I is None else I
        I[:] = np.arange(k)
        return D, I


def test_one_at_a_time_counts_warmup_and_repeats():
    index, queries = CountingIndex(), np.zeros((50, 4), dtype=np.float32)
    result = time_one_at_a_time(index, queries, repeats=3, warmup=5)
    assert index.calls == 5 + 3 * 50
    assert len(result["times"]) == 3 and result["ids"].shape == (50, K)
    assert result["qps"] == pytest.approx(50 / np.median(result["times"]))


def test_batch_timing_is_one_call_per_pass():
    index = CountingIndex()
    result = time_batch(index, np.zeros((50, 4), dtype=np.float32), repeats=3)
    assert index.calls == 1 + 3 and len(result["times"]) == 3


@pytest.mark.skipif(not hasattr(os, "sched_getaffinity"), reason="no CPU affinity on this platform")
def test_single_thread_pinning_is_undone_afterwards():
    threads_before, cpus_before = faiss.omp_get_max_threads(), os.sched_getaffinity(0)
    with single_thread_pinned():
        assert faiss.omp_get_max_threads() == 1
        assert os.sched_getaffinity(0) == {TIMING_CPU}
    assert faiss.omp_get_max_threads() == threads_before
    assert os.sched_getaffinity(0) == cpus_before


def test_distance_counters_cover_only_the_timed_passes():
    base, queries = random_data(n=2_000)
    index = faiss.index_factory(base.shape[1], "IVF10,Flat")
    index.train(base)
    index.add(base)
    index.nprobe = 10  # every list: each search scans every vector once
    result = time_one_at_a_time(index, queries, repeats=2, warmup=5)
    assert result["counted_per_search"] == len(base)
    assert time_one_at_a_time(CountingIndex(), queries, repeats=1, warmup=0)["counted_per_search"] == 0


def test_one_at_a_time_ids_match_a_batch_search():
    base, queries = random_data()
    index = faiss.IndexFlatL2(base.shape[1])
    index.add(base)
    result = time_one_at_a_time(index, queries, repeats=1, warmup=0)
    assert (result["ids"] == index.search(queries, K)[1]).all()

## Indexes
All four indexes come from FAISS. k-means seeds are fixed. Build time is the wall-clock time of train + add with `BUILD_THREADS` threads. Index size is the size of the serialised index, which is held in memory and never written to disk. IVF-PQ also keeps a precomputed distance table in RAM that FAISS rebuilds on load instead of storing; `search_table_bytes` records it separately.

In [ ]:
SEARCH_PARAM = {"exact": "", "hnsw": "efSearch", "ivf_flat": "nprobe", "ivf_pq": "nprobe"}


def make_index(kind, dim, **params):
    """An empty FAISS index: 'exact', 'hnsw' (M, efC), 'ivf_flat' (nlist) or 'ivf_pq' (nlist, m)."""
    if kind == "exact":
        return faiss.IndexFlatL2(dim)
    if kind == "hnsw":
        index = faiss.index_factory(dim, f"HNSW{params['M']}")
        index.hnsw.efConstruction = params["efC"]
        return index
    if kind == "ivf_flat":
        index = faiss.index_factory(dim, f"IVF{params['nlist']},Flat")
    elif kind == "ivf_pq":
        index = faiss.index_factory(dim, f"IVF{params['nlist']},PQ{params['m']}")
        index.pq.cp.seed = SEED
    else:
        raise ValueError(f"unknown index kind {kind!r}")
    index.cp.seed = SEED
    return index


def set_search_param(index, kind, value):
    if kind == "hnsw":
        index.hnsw.efSearch = int(value)
    elif kind in ("ivf_flat", "ivf_pq"):
        index.nprobe = int(value)


def build_index(kind, base, **params):
    """Train (if needed) and add `base`; returns the index and its wall-clock build time in seconds."""
    index = make_index(kind, base.shape[1], **params)
    with faiss_threads(BUILD_THREADS), threadpoolctl.threadpool_limits(BUILD_THREADS):
        start = time.perf_counter()
        if not index.is_trained:
            index.train(base)
        index.add(base)
        build_time = time.perf_counter() - start
    return index, build_time


def index_bytes(index):
    return int(faiss.serialize_index(index).nbytes)


def distances_per_query(index, counted_per_search):
    """Distance computations per query: HNSW's own count; for IVF, the list entries scanned plus the nlist centroid
    distances of the coarse quantiser (FAISS does not count those); for exact search, every vector."""
    inner = faiss.downcast_index(index)
    if isinstance(inner, faiss.IndexFlat):
        return float(inner.ntotal)
    if isinstance(inner, faiss.IndexIVF):
        return counted_per_search + inner.nlist
    return float(counted_per_search)


def search_table_bytes(index):
    """RAM that IVF-PQ keeps beside its serialised form: the precomputed distance table (nlist x m x 256 float32)."""
    inner = faiss.downcast_index(index)
    if isinstance(inner, faiss.IndexIVFPQ) and inner.use_precomputed_table:
        return int(inner.precomputed_table.size()) * 4
    return 0

In [ ]:
%%ipytest -qq

@pytest.mark.parametrize("kind, params", [
    ("exact", {}), ("hnsw", dict(M=8, efC=40)), ("ivf_flat", dict(nlist=8)), ("ivf_pq", dict(nlist=8, m=4)),
])
def test_every_index_kind_builds_and_searches(kind, params):
    base, queries = random_data(n=10_000, d=8)
    index, build_time = build_index(kind, base, **params)
    set_search_param(index, kind, 16)
    _, ids = index.search(queries, K)
    assert index.ntotal == len(base) and build_time > 0 and index_bytes(index) > 0
    assert ids.shape == (len(queries), K) and (ids >= 0).all()


def test_kmeans_seeds_are_fixed():
    assert make_index("ivf_flat", 8, nlist=8).cp.seed == SEED
    pq = make_index("ivf_pq", 8, nlist=8, m=4)
    assert pq.cp.seed == SEED and pq.pq.cp.seed == SEED


def test_search_parameters_are_applied():
    hnsw = make_index("hnsw", 8, M=8, efC=40)
    set_search_param(hnsw, "hnsw", 77)
    ivf = make_index("ivf_flat", 8, nlist=8)
    set_search_param(ivf, "ivf_flat", 5)
    assert hnsw.hnsw.efSearch == 77 and ivf.nprobe == 5


def test_only_ivf_pq_keeps_a_search_table():
    base, _ = random_data(n=10_000, d=8)
    assert search_table_bytes(build_index("ivf_pq", base, nlist=8, m=4)[0]) == 8 * 4 * 256 * 4
    assert search_table_bytes(build_index("ivf_flat", base, nlist=8)[0]) == 0
    assert search_table_bytes(build_index("hnsw", base, M=8, efC=40)[0]) == 0


def test_distances_per_query_by_index_kind():
    base, _ = random_data(n=2_000)
    assert distances_per_query(build_index("exact", base)[0], 0) == len(base)
    assert distances_per_query(build_index("ivf_flat", base, nlist=8)[0], 100.0) == 108
    assert distances_per_query(build_index("hnsw", base, M=8, efC=40)[0], 55.5) == 55.5


def test_index_size_grows_with_the_number_of_vectors():
    base, _ = random_data(n=2_000)
    assert index_bytes(build_index("exact", base)[0]) > index_bytes(build_index("exact", base[:500])[0])

## Benchmark runs
`benchmark_index` builds one configuration, records its build cost, then sweeps the search parameter. Every setting is measured on both query halves, and the sweep stops once the tune-half recall reaches `recall_stop`. `benchmark_exact` times exact search on the exact-timing queries. Every row carries the run's settings and the machine record.

In [ ]:
def run_header(part, section, data, kind, params, build_time, size, variant="", table_bytes=0):
    return dict(
        part=part, section=section, dataset=data.name, variant=variant, n_base=len(data.base), dim=data.dim,
        index=kind, build_params=json.dumps(params, sort_keys=True), build_time_s=round(build_time, 4),
        index_bytes=size, bytes_per_vector=size / len(data.base), search_table_bytes=table_bytes, search_threads=1,
        mode=MODE, seed=SEED, timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"), load_avg_1m=os.getloadavg()[0], **MACHINE,
    )


def measure(index, data, gt, split, rows=None):
    """Recall@10 and single-thread QPS on one query half ('tune' or 'report'), or on explicit query rows."""
    rows = (TUNE if split == "tune" else REPORT) if rows is None else rows
    queries = data.queries[rows]
    timing = time_one_at_a_time(index, queries)
    return dict(
        split=split, n_queries=len(rows),
        recall_at_10=recall_at_k(timing["ids"], data.base, queries, gt[rows]),
        qps=timing["qps"], times_s=json.dumps([round(t, 6) for t in timing["times"]]),
        distances_per_query=distances_per_query(index, timing["counted_per_search"]),
    )


def run_sweep(index, kind, values, data, gt, header, halves=None):
    halves = halves or {"tune": TUNE, "report": REPORT}
    rows = []
    for value in values:
        set_search_param(index, kind, value)
        for split, query_rows in halves.items():
            rows.append({**header, "param_name": SEARCH_PARAM[kind], "param_value": value,
                         **measure(index, data, gt, split, query_rows)})
        tune_recall = next(r["recall_at_10"] for r in rows[::-1] if r["split"] == "tune")
        if tune_recall >= CONFIG["recall_stop"]:
            break
    return rows


def benchmark_index(part, section, data, gt, kind, params, values, variant="", halves=None):
    """Build one configuration, record its build time and size, then sweep its search parameter."""
    index, build_time = build_index(kind, data.base, **params)
    header = run_header(part, section, data, kind, params, build_time, index_bytes(index), variant,
                        search_table_bytes(index))
    rows = run_sweep(index, kind, values, data, gt, header, halves)
    best = max(r["recall_at_10"] for r in rows if r["split"] == "tune")
    print(f"{section} {variant} {kind} {params}: built in {build_time:.1f} s, best tune recall {best:.3f}")
    return rows


def benchmark_exact(part, section, data, gt, variant="", rows=None):
    """Exact search timed one query at a time on the exact-timing queries; recall is 1 by construction."""
    index, build_time = build_index("exact", data.base)
    header = run_header(part, section, data, "exact", {}, build_time, index_bytes(index), variant)
    rows = np.arange(*CONFIG["exact_timing"]) if rows is None else rows
    return [{**header, "param_name": "", "param_value": 0, **measure(index, data, gt, "report", rows)}]

In [ ]:
%%ipytest -qq

def toy_dataset(n=3_000, nq=60):
    base, queries = random_data(n=n, d=8, nq=nq)
    return Dataset("toy", base, queries, None, None), exact_knn(base, queries, 20)[1]


def test_sweep_measures_both_halves_and_stops_once_tune_recall_is_high():
    data, gt = toy_dataset()
    halves = {"tune": np.arange(30), "report": np.arange(30, 60)}
    rows = benchmark_index(0, "test", data, gt, "hnsw", dict(M=8, efC=40), [10, 16, 32, 64, 128, 256], halves=halves)
    assert [r["split"] for r in rows] == ["tune", "report"] * (len(rows) // 2)
    tune = [r["recall_at_10"] for r in rows if r["split"] == "tune"]
    assert all(r < CONFIG["recall_stop"] for r in tune[:-1])
    assert all(0 <= r <= 1 for r in tune) and rows[0]["param_name"] == "efSearch"


def test_exact_search_has_perfect_recall():
    data, gt = toy_dataset()
    [row] = benchmark_exact(0, "test", data, gt, rows=np.arange(30, 60))
    assert row["recall_at_10"] == 1.0 and row["index"] == "exact" and row["split"] == "report"
    assert row["distances_per_query"] == len(data.base) and row["load_avg_1m"] >= 0

## Choosing configurations (tune half only)
Pareto fronts and operating points are picked from **tune-half** rows only. The reported numbers are the matching **report-half** rows, even when those land just under a target.

In [ ]:
CONFIG_KEYS = ["part", "section", "dataset", "variant", "n_base", "index", "build_params", "param_name", "param_value"]


def pareto_front(rows):
    """Tune-half rows that no other tune-half row beats on both recall and QPS, sorted by recall."""
    tune = rows[rows["split"] == "tune"].sort_values(["recall_at_10", "qps"], ascending=[False, False])
    keep, best_qps = [], -np.inf
    for label, row in tune.iterrows():
        if row["qps"] > best_qps:
            keep.append(label)
            best_qps = row["qps"]
    return tune.loc[keep].sort_values("recall_at_10")


def report_rows_for(chosen, rows):
    """The report-half rows measured with the same configurations as `chosen` (tune-half rows)."""
    report = rows[rows["split"] == "report"]
    return chosen[CONFIG_KEYS].merge(report, on=CONFIG_KEYS, how="left")


def pick_operating_point(rows, target):
    """The fastest configuration whose tune-half recall reaches `target`, as its report-half row (None if none does)."""
    reached = rows[(rows["split"] == "tune") & (rows["recall_at_10"] >= target)]
    if reached.empty:
        return None
    return report_rows_for(reached.loc[[reached["qps"].idxmax()]], rows).iloc[0]

In [ ]:
%%ipytest -qq

def toy_rows():
    """One index, five efSearch values: (tune recall, tune QPS, report recall, report QPS)."""
    common = dict(part=2, section="toy", dataset="toy", variant="", n_base=100, index="hnsw",
                  build_params="{}", param_name="efSearch")
    measured = {16: (0.80, 1000, 0.79, 990), 32: (0.92, 600, 0.90, 610), 48: (0.91, 500, 0.95, 490),
                64: (0.96, 300, 0.94, 305), 128: (0.99, 150, 0.99, 140)}
    rows = []
    for value, (tr, tq, rr, rq) in measured.items():
        rows.append(dict(common, param_value=value, split="tune", recall_at_10=tr, qps=tq))
        rows.append(dict(common, param_value=value, split="report", recall_at_10=rr, qps=rq))
    return pd.DataFrame(rows)


def test_pareto_front_drops_dominated_settings():
    assert pareto_front(toy_rows())["param_value"].tolist() == [16, 32, 64, 128]  # 48 is beaten by 32


def test_operating_point_is_chosen_on_the_tune_half_and_reported_on_the_report_half():
    at_090 = pick_operating_point(toy_rows(), 0.90)
    assert at_090["param_value"] == 32 and at_090["recall_at_10"] == 0.90 and at_090["qps"] == 610
    at_095 = pick_operating_point(toy_rows(), 0.95)
    # The report half alone would suggest 48 (0.95 at 490 QPS); choosing on it would be tuning on reported queries.
    assert at_095["param_value"] == 64 and at_095["recall_at_10"] == 0.94
    assert pick_operating_point(toy_rows(), 0.999) is None

## Results file
Rows are grouped by `(part, section)`. Rerunning a section replaces exactly its own rows, and the file is written atomically.

In [ ]:
def write_rows(rows, part, section, results_dir=None):
    path = Path(results_dir or RESULTS_DIR) / "results.csv"
    table = pd.DataFrame(rows).assign(part=part, section=section)
    if path.exists():
        old = pd.read_csv(path)
        mine = (old["part"].astype(str) == str(part)) & (old["section"].astype(str) == str(section))
        table = pd.concat([old[~mine], table], ignore_index=True)
    temporary = path.with_suffix(".tmp")
    table.to_csv(temporary, index=False)
    temporary.replace(path)
    return path


def read_results(results_dir=None):
    table = pd.read_csv(Path(results_dir or RESULTS_DIR) / "results.csv")
    for column in ("variant", "param_name", "method"):  # empty text is written as "" and read back as NaN
        if column in table:
            table[column] = table[column].astype("str").fillna("")
    return table

In [ ]:
%%ipytest -qq

def test_write_rows_replaces_only_its_own_section(tmp_path):
    write_rows([dict(x=1), dict(x=2)], part=2, section="sift", results_dir=tmp_path)
    write_rows([dict(x=3)], part=2, section="glove", results_dir=tmp_path)
    write_rows([dict(x=4)], part=2, section="sift", results_dir=tmp_path)
    assert sorted(read_results(tmp_path)["x"]) == [3, 4]
    assert not list(tmp_path.glob("*.tmp"))


def test_empty_text_fields_survive_a_round_trip(tmp_path):
    write_rows([dict(variant="", param_name="", x=1)], part=1, section="s", results_dir=tmp_path)
    row = read_results(tmp_path).iloc[0]
    assert row["variant"] == "" and row["param_name"] == ""

## Figures and tables
Every figure and table is drawn from `results.csv` alone, by the functions below. The part notebooks call them right after their runs, and `05_report_assets.ipynb` calls them again, so a figure can be redrawn without re-running a benchmark.

Each index keeps one colour and one marker in every figure. The three index colours are the first three slots of a categorical palette that passes a colour-blindness separation check for every pair; exact search is drawn in neutral grey. The markers repeat the identity for greyscale print. Figures are saved as PDF (for the report) and PNG; tables as Markdown and LaTeX.

In [ ]:
INK = dict(primary="#0b0b0b", secondary="#52514e", muted="#898781", grid="#e1e0d9", axis="#c3c2b7")
INDEX_STYLE = {
    "hnsw": dict(label="HNSW", color="#2a78d6", marker="o"),
    "ivf_flat": dict(label="IVF-Flat", color="#eb6834", marker="s"),
    "ivf_pq": dict(label="IVF-PQ", color="#1baf7a", marker="^"),
    "exact": dict(label="Exact search", color=INK["secondary"], marker="D"),
}
DATASET_LABEL = {"sift": "SIFT", "glove": "GloVe-100"}
STYLE = {
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 8, "axes.titlesize": 9, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlecolor": INK["primary"], "axes.labelcolor": INK["secondary"], "text.color": INK["primary"],
    "axes.edgecolor": INK["axis"], "axes.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": INK["grid"], "grid.linewidth": 0.6,
    "xtick.color": INK["axis"], "ytick.color": INK["axis"], "xtick.labelcolor": INK["secondary"],
    "ytick.labelcolor": INK["secondary"], "lines.linewidth": 1.5, "lines.markersize": 4.5,
    "lines.solid_joinstyle": "round", "lines.solid_capstyle": "round", "legend.frameon": False,
    "legend.fontsize": 8, "pdf.fonttype": 42, "savefig.dpi": 200, "savefig.bbox": "tight",
}


def compact(value, _=None):
    """Axis tick text: 1500 -> '1.5K', 20000 -> '20K'."""
    for limit, suffix in ((1e6, "M"), (1e3, "K")):
        if abs(value) >= limit:
            return f"{value / limit:g}{suffix}"
    return f"{value:g}"


def log_ticks(axis):
    """Labelled ticks at 1 and 3 times each power of ten, in compact text, on a log axis."""
    axis.set_major_locator(ticker.LogLocator(subs=(1.0, 3.0)))
    axis.set_major_formatter(ticker.FuncFormatter(compact))
    axis.set_minor_formatter(ticker.NullFormatter())


def format_qps(qps):
    return f"{qps:,.0f}" if qps >= 100 else f"{qps:.1f}"


def save_figure(fig, name, figures_dir=None):
    """Writes <name>.pdf and <name>.png; the PDF carries no creation date, so an unchanged figure gives an unchanged file."""
    directory = Path(figures_dir or FIGURES_DIR)
    with plt.rc_context(STYLE):
        fig.savefig(directory / f"{name}.pdf", metadata={"CreationDate": None})
        fig.savefig(directory / f"{name}.png")
    return [directory / f"{name}.pdf", directory / f"{name}.png"]


def pretty_table(table, columns):
    """Report version of a table: `columns` maps each column to (header, formatter); missing values become ''."""
    return pd.DataFrame({
        header: [("" if pd.isna(value) else formatter(value)) for value in table[column]]
        for column, (header, formatter) in columns.items()
    })


def markdown_table(table):
    lines = ["| " + " | ".join(map(str, row)) + " |" for row in [list(table.columns), *table.values.tolist()]]
    lines.insert(1, "|" + "|".join("---" for _ in table.columns) + "|")
    return "\n".join(lines) + "\n"


def latex_table(table):
    """A booktabs tabular; columns holding only numbers are right-aligned."""
    escape = lambda text: re.sub(r"([&%$#_{}])", r"\\\1", str(text))
    numeric = [table[column].astype(str).str.fullmatch(r"[\d.,\-]*").all() for column in table.columns]
    lines = [r"\begin{tabular}{" + "".join("r" if n else "l" for n in numeric) + "}", r"\toprule",
             " & ".join(map(escape, table.columns)) + r" \\", r"\midrule",
             *(" & ".join(map(escape, row)) + r" \\" for row in table.values.tolist()),
             r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines) + "\n"


def write_table(table, name, tables_dir=None):
    directory = Path(tables_dir or TABLES_DIR)
    (directory / f"{name}.md").write_text(markdown_table(table))
    (directory / f"{name}.tex").write_text(latex_table(table))
    return table


def direct_labels(ax, labels):
    """Places (text, xy, offset, alignment) labels in label ink; one that would overlap a placed label or leave the axes is
    dropped, and the legend still names its series."""
    renderer = ax.figure.canvas.get_renderer()
    frame, placed = ax.get_window_extent(renderer), []
    for text, xy, offset, ha, va, xycoords in labels:
        label = ax.annotate(text, xy, xycoords=xycoords, xytext=offset, textcoords="offset points", ha=ha, va=va,
                            color=INK["secondary"], fontsize=7)
        box = label.get_window_extent(renderer)
        inside = frame.x0 <= box.x0 and box.x1 <= frame.x1 and frame.y0 <= box.y0 and box.y1 <= frame.y1
        if inside and not any(box.overlaps(other) for other in placed):
            placed.append(box)
        else:
            label.remove()

In [ ]:
%%ipytest -qq

def test_compact_tick_text():
    assert [compact(v) for v in (10, 999, 1_000, 1_500, 20_000, 3_000_000)] == ["10", "999", "1K", "1.5K", "20K", "3M"]
    assert compact(0.03) == "0.03"


def test_pretty_table_formats_and_blanks_missing_values():
    table = pd.DataFrame(dict(a=["x", "y"], b=[1234.5, np.nan]))
    pretty = pretty_table(table, {"a": ("Name", str), "b": ("QPS", format_qps)})
    assert pretty.to_dict("list") == {"Name": ["x", "y"], "QPS": ["1,234", ""]}


def test_table_writers(tmp_path):
    table = pd.DataFrame({"Index": ["IVF_PQ"], "QPS": ["1,234"]})
    write_table(table, "t", tables_dir=tmp_path)
    assert (tmp_path / "t.md").read_text() == "| Index | QPS |\n|---|---|\n| IVF_PQ | 1,234 |\n"
    tex = (tmp_path / "t.tex").read_text()
    assert r"\begin{tabular}{lr}" in tex and r"IVF\_PQ & 1,234 \\" in tex


def test_save_figure_writes_pdf_and_png(tmp_path):
    fig, ax = plt.subplots()
    ax.plot([0, 1], [0, 1])
    paths = save_figure(fig, "f", figures_dir=tmp_path)
    plt.close(fig)
    assert all(path.exists() and path.stat().st_size > 0 for path in paths)


def test_overlapping_direct_labels_are_dropped():
    fig, ax = plt.subplots()
    ax.set(xlim=(0, 1), ylim=(0, 1))
    fig.canvas.draw()
    direct_labels(ax, [("first", (0.5, 0.5), (0, 0), "center", "center", "data"),
                       ("second", (0.5, 0.5), (0, 0), "center", "center", "data"),
                       ("outside", (1.0, 0.5), (0, 0), "left", "center", "data")])
    assert [t.get_text() for t in ax.texts] == ["first"]
    plt.close(fig)

### Part 2: recall against speed, and the two tables
- **Best trade-off curve** of an index: the Pareto front of its tune-half rows over all build settings and search values, drawn at the report-half recall and QPS of the same configurations. Faint dots are every configuration measured (report half).
- **Exact baseline:** Part 1's one-at-a-time row, timed with the same protocol on the same base set.
- **Operating point** at a target recall: the fastest configuration whose tune-half recall reaches it, reported on the report half (`pick_operating_point`). An index that never reaches 0.95 on the tune half is shown in the cost table at its highest-recall configuration, marked as not reaching it.

In [ ]:
PART2_INDEXES = ["hnsw", "ivf_flat", "ivf_pq"]
BUILD_PARAM_ORDER = ["M", "nlist", "m"]  # efConstruction is the same for every HNSW build, so labels leave it out


def describe_setting(row, search=True):
    """A configuration as text, e.g. 'M=16, efSearch=64' or 'nlist=4096, m=32, nprobe=8'."""
    params = json.loads(row["build_params"])
    parts = [f"{key}={params[key]}" for key in BUILD_PARAM_ORDER if key in params]
    if search and row["param_name"]:
        parts.append(f"{row['param_name']}={int(row['param_value'])}")
    return ", ".join(parts) or "full scan"


def part2_rows(results, dataset):
    return results[(results["part"] == 2) & (results["dataset"] == dataset)]


def part2_datasets(results):
    """Datasets with Part 2 rows, SIFT first, then GloVe, then any others."""
    present = dict.fromkeys(results.loc[results["part"] == 2, "dataset"])
    known = list(DATASET_LABEL)
    return sorted(present, key=lambda name: known.index(name) if name in known else len(known))


def exact_baseline(results, dataset):
    """Part 1's one-at-a-time exact-search row for `dataset` (None if Part 1 has not run)."""
    if "method" not in results:
        return None
    rows = results[(results["part"] == 1) & (results["dataset"] == dataset) & (results["method"] == "one_at_a_time")]
    return rows.iloc[-1] if len(rows) else None


def tradeoff_curve(rows):
    """An index's best trade-off curve: report-half rows of its tune-half Pareto front, in order of tune recall."""
    return report_rows_for(pareto_front(rows), rows)


def highest_recall_setting(rows):
    """Report-half row of the configuration with the highest tune-half recall (ties go to the faster one)."""
    tune = rows[rows["split"] == "tune"].sort_values(["recall_at_10", "qps"], ascending=False)
    return report_rows_for(tune.iloc[:1], rows).iloc[0]


def plot_tradeoff(ax, results, dataset):
    """Draws one dataset's panel; returns the direct labels to place once the figure layout is final."""
    rows = part2_rows(results, dataset)
    labels = []
    for kind in PART2_INDEXES:
        mine = rows[rows["index"] == kind]
        if mine.empty:
            continue
        style, every, curve = INDEX_STYLE[kind], mine[mine["split"] == "report"], tradeoff_curve(mine)
        ax.scatter(every["recall_at_10"], every["qps"], s=7, color=style["color"], alpha=0.3, linewidths=0, zorder=2)
        ax.plot(curve["recall_at_10"], curve["qps"], color=style["color"], marker=style["marker"], label=style["label"],
                markeredgecolor="white", markeredgewidth=0.8, zorder=3)
        end = (curve["recall_at_10"].iloc[-1], curve["qps"].iloc[-1])
        labels.append((style["label"], end, (-5, -4), "right", "top", "data"))
    exact = exact_baseline(results, dataset)
    if exact is not None:
        assert exact["n_base"] == rows["n_base"].iloc[0], "Part 1 and Part 2 timed different base sets"
        style = INDEX_STYLE["exact"]
        ax.axhline(exact["qps"], color=style["color"], linewidth=0.8, zorder=1)
        ax.plot([exact["recall_at_10"]], [exact["qps"]], linestyle="none", color=style["color"], marker=style["marker"],
                label=style["label"], markeredgecolor="white", markeredgewidth=0.8, zorder=3)
        labels.append((style["label"], (0.01, exact["qps"]), (0, 2), "left", "bottom", ("axes fraction", "data")))
    for target, offset, ha in ((0.90, -2, "right"), (0.95, 2, "left")):
        ax.axvline(target, color=INK["axis"], linewidth=0.6, zorder=1)
        labels.append((f"{target:.2f}", (target, 1), (offset, -2), ha, "top", ("data", "axes fraction")))
    ax.set_yscale("log")
    log_ticks(ax.yaxis)
    ax.set_xlabel("Recall@10")
    ax.set_ylabel("Queries per second (1 thread)")
    ax.set_title(f"{DATASET_LABEL.get(dataset, dataset)}, {int(rows['n_base'].iloc[0]):,} vectors")
    return labels


def part2_figure(results):
    datasets = part2_datasets(results)
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, len(datasets), figsize=(3.4 * len(datasets), 3.0), layout="constrained", squeeze=False)
        placements = [plot_tradeoff(ax, results, dataset) for ax, dataset in zip(axes[0], datasets)]
        handles = {}
        for ax in axes[0]:
            for handle, label in zip(*ax.get_legend_handles_labels()):
                handles.setdefault(label, handle)
        fig.legend(handles.values(), handles.keys(), loc="outside upper center", ncols=len(handles))
        fig.canvas.draw()
        for ax, labels in zip(axes[0], placements):
            direct_labels(ax, labels)
    return fig


def qps_at_recall_table(results, targets=(0.90, 0.95)):
    """Per dataset and index, the setting chosen on the tune half for each target, with its report-half recall and QPS."""
    records = []
    for dataset in part2_datasets(results):
        rows = part2_rows(results, dataset)
        for kind in PART2_INDEXES:
            mine = rows[rows["index"] == kind]
            if mine.empty:
                continue
            record = dict(dataset=dataset, index=kind)
            for target in targets:
                point = pick_operating_point(mine, target)
                record[f"setting_{target:.2f}"] = "not reached" if point is None else describe_setting(point)
                record[f"recall_{target:.2f}"] = np.nan if point is None else point["recall_at_10"]
                record[f"qps_{target:.2f}"] = np.nan if point is None else point["qps"]
            records.append(record)
        exact = exact_baseline(results, dataset)
        if exact is not None:
            record = dict(dataset=dataset, index="exact")
            for target in targets:
                record.update({f"setting_{target:.2f}": "full scan", f"recall_{target:.2f}": exact["recall_at_10"],
                               f"qps_{target:.2f}": exact["qps"]})
            records.append(record)
    return pd.DataFrame(records)


def cost_at_recall_table(results, target=0.95):
    """Build time, size and bytes per vector of each index's operating point at `target` (or its highest-recall
    configuration if it never reaches `target` on the tune half), with exact search for reference."""
    records = []
    for dataset in part2_datasets(results):
        rows = part2_rows(results, dataset)
        chosen = [(kind, pick_operating_point(rows[rows["index"] == kind], target)) for kind in PART2_INDEXES
                  if (rows["index"] == kind).any()]
        chosen = [(kind, True, point) if point is not None else
                  (kind, False, highest_recall_setting(rows[rows["index"] == kind])) for kind, point in chosen]
        exact = exact_baseline(results, dataset)
        if exact is not None:
            chosen.append(("exact", True, exact))
        for kind, reached, point in chosen:
            records.append(dict(
                dataset=dataset, index=kind, setting=describe_setting(point), reaches_target=reached,
                recall_at_10=point["recall_at_10"], qps=point["qps"], build_time_s=point["build_time_s"],
                index_mb=point["index_bytes"] / 1e6, bytes_per_vector=point["bytes_per_vector"],
                search_table_mb=(0 if pd.isna(point.get("search_table_bytes")) else point["search_table_bytes"]) / 1e6,
            ))
    return pd.DataFrame(records)


def best_recall_table(results):
    """Highest tune-half recall each build setting reached in its sweep, and the report-half recall there."""
    records = []
    for dataset in part2_datasets(results):
        rows = part2_rows(results, dataset)
        for kind in PART2_INDEXES:
            for _, mine in rows[rows["index"] == kind].groupby("build_params", sort=False):
                best, tune = highest_recall_setting(mine), mine[mine["split"] == "tune"]
                records.append(dict(dataset=dataset, index=kind, build=describe_setting(best, search=False),
                                    search=f"{best['param_name']}={int(best['param_value'])}",
                                    largest_swept=int(tune["param_value"].max()), tune_recall=tune["recall_at_10"].max(),
                                    report_recall=best["recall_at_10"]))
    return pd.DataFrame(records)


NAME = dict(dataset=lambda d: DATASET_LABEL.get(d, d), index=lambda k: INDEX_STYLE[k]["label"])
RECALL = "{:.3f}".format


def part2_assets(results):
    """Draws the Part 2 figure and writes its tables; returns the figure and the report versions of the tables."""
    fig = part2_figure(results)
    save_figure(fig, "part2_recall_qps")
    tables = {
        "part2_qps_at_recall": pretty_table(qps_at_recall_table(results), {
            "dataset": ("Dataset", NAME["dataset"]), "index": ("Index", NAME["index"]),
            "setting_0.90": ("Setting for 0.90", str), "recall_0.90": ("Recall (0.90)", RECALL),
            "qps_0.90": ("QPS (0.90)", format_qps),
            "setting_0.95": ("Setting for 0.95", str), "recall_0.95": ("Recall (0.95)", RECALL),
            "qps_0.95": ("QPS (0.95)", format_qps)}),
        "part2_cost_at_095": pretty_table(cost_at_recall_table(results), {
            "dataset": ("Dataset", NAME["dataset"]), "index": ("Index", NAME["index"]), "setting": ("Setting", str),
            "reaches_target": ("Reaches 0.95", lambda r: "yes" if r else "no"), "recall_at_10": ("Recall", RECALL),
            "qps": ("QPS", format_qps), "build_time_s": ("Build (s)", "{:.1f}".format),
            "index_mb": ("Size (MB)", "{:.1f}".format), "bytes_per_vector": ("Bytes/vector", "{:.0f}".format),
            "search_table_mb": ("Search table (MB)", "{:.1f}".format)}),
        "part2_best_recall": pretty_table(best_recall_table(results), {
            "dataset": ("Dataset", NAME["dataset"]), "index": ("Index", NAME["index"]), "build": ("Build", str),
            "search": ("Best search setting", str), "largest_swept": ("Largest swept", str),
            "tune_recall": ("Tune recall", RECALL), "report_recall": ("Report recall", RECALL)}),
    }
    for name, table in tables.items():
        write_table(table, name)
    return fig, tables

In [ ]:
%%ipytest -qq

def toy_results():
    """Part 1 exact row plus Part 2 sweeps: HNSW (two builds) reaches 0.95, IVF-PQ tops out at 0.90."""
    common = dict(section="toy", dataset="toy", variant="", n_base=1000, method="", build_time_s=1.0,
                  index_bytes=8000, bytes_per_vector=8.0, search_table_bytes=0)
    rows = [dict(common, part=1, index="exact", build_params="{}", param_name="", param_value=0, split="report",
                 recall_at_10=1.0, qps=50.0, method="one_at_a_time")]
    sweeps = {("hnsw", '{"M": 8, "efC": 200}', "efSearch"): {16: (0.85, 900), 32: (0.93, 450), 64: (0.97, 180)},
              ("hnsw", '{"M": 16, "efC": 200}', "efSearch"): {16: (0.91, 700), 32: (0.96, 500), 64: (0.99, 200)},
              ("ivf_pq", '{"m": 4, "nlist": 16}', "nprobe"): {1: (0.70, 2000), 4: (0.88, 1500), 16: (0.90, 800)}}
    for (kind, params, param_name), measured in sweeps.items():
        for value, (recall, qps) in measured.items():
            for split, shift in (("tune", 0.0), ("report", -0.005)):
                rows.append(dict(common, part=2, index=kind, build_params=params, param_name=param_name,
                                 param_value=value, split=split, recall_at_10=recall + shift, qps=qps))
    return pd.DataFrame(rows)


def test_describe_setting():
    row = dict(build_params='{"m": 32, "nlist": 4096}', param_name="nprobe", param_value=8)
    assert describe_setting(row) == "nlist=4096, m=32, nprobe=8"
    assert describe_setting(dict(build_params='{"M": 16, "efC": 200}', param_name="efSearch", param_value=64)) == "M=16, efSearch=64"
    assert describe_setting(dict(build_params="{}", param_name="", param_value=0)) == "full scan"


def test_qps_table_picks_on_the_tune_half_across_build_settings():
    table = qps_at_recall_table(toy_results()).set_index("index")
    assert table.loc["hnsw", "setting_0.90"] == "M=16, efSearch=16"   # 700 QPS beats M=8's 450 at tune recall >= 0.90
    assert table.loc["hnsw", "setting_0.95"] == "M=16, efSearch=32"
    assert table.loc["hnsw", "recall_0.95"] == pytest.approx(0.955)  # the report-half value
    assert table.loc["ivf_pq", "setting_0.95"] == "not reached" and np.isnan(table.loc["ivf_pq", "qps_0.95"])
    assert table.loc["exact", "qps_0.95"] == 50.0


def test_cost_table_marks_an_index_that_never_reaches_the_target():
    table = cost_at_recall_table(toy_results()).set_index("index")
    assert table.loc["hnsw", "reaches_target"] and not table.loc["ivf_pq", "reaches_target"]
    assert table.loc["ivf_pq", "setting"] == "nlist=16, m=4, nprobe=16"
    assert table.loc["exact", "setting"] == "full scan"


def test_best_recall_table_has_one_row_per_build():
    table = best_recall_table(toy_results())
    assert table["build"].tolist() == ["M=8", "M=16", "nlist=16, m=4"]
    assert table["tune_recall"].tolist() == [0.97, 0.99, 0.90]


def test_figure_draws_every_index_and_the_exact_baseline():
    fig = part2_figure(toy_results())
    [ax] = fig.axes
    assert [line.get_label() for line in ax.get_lines() if not line.get_label().startswith("_")] == ["HNSW", "IVF-PQ", "Exact search"]
    hnsw = next(line for line in ax.get_lines() if line.get_label() == "HNSW")
    assert list(hnsw.get_xdata()) == pytest.approx([0.845, 0.905, 0.955, 0.985])  # tune front, report values
    plt.close(fig)

### Part 3: how the cost of recall 0.95 grows with N
- **Cost at each size:** for each index, the operating point at 0.95 (chosen on the tune half, reported on the report half): latency (1000 / QPS), distance computations per query, build time and bytes per vector. Exact search is its one-at-a-time row.
- **Growth rate:** the slope of a least-squares line through log(cost) against log(N). A slope of 1 means cost grows like N, 0.5 like √N, and close to 0 like log N.
- **Q4 inputs:** the same fit gives latency at 150K (inside the measured range) and at 10M (an extrapolation); memory at 10M is bytes per vector at the largest size × 10M.

In [ ]:
SCALING_METHODS = ["exact", "ivf_flat", "hnsw"]
Q4_SIZES = [150_000, 10_000_000]


def scaling_nlist(n):
    """IVF-Flat lists for N vectors: N / nlist = sqrt(N) / 4 vectors per list on average."""
    return int(round(4 * np.sqrt(n)))


def scaling_table(results, target=0.95):
    """Per base size and method, the operating point at `target` (exact search: its one-at-a-time row)."""
    part3, records = results[results["part"] == 3], []
    for n in sorted(part3["n_base"].unique()):
        rows = part3[part3["n_base"] == n]
        for kind in SCALING_METHODS:
            mine = rows[rows["index"] == kind]
            if mine.empty:
                continue
            point = mine.iloc[0] if kind == "exact" else pick_operating_point(mine, target)
            if point is None:
                records.append(dict(n_base=int(n), index=kind, setting="not reached"))
                continue
            records.append(dict(
                n_base=int(n), index=kind, setting=describe_setting(point), recall_at_10=point["recall_at_10"],
                qps=point["qps"], latency_ms=1000 / point["qps"], distances_per_query=point["distances_per_query"],
                build_time_s=point["build_time_s"], bytes_per_vector=point["bytes_per_vector"],
            ))
    return pd.DataFrame(records)


def power_law(n, values):
    """Least-squares fit of log(value) = log(a) + b log(n); returns (a, b)."""
    b, log_a = np.polyfit(np.log(np.asarray(n, float)), np.log(np.asarray(values, float)), 1)
    return float(np.exp(log_a)), float(b)


def scaling_fit_table(scaling, at=Q4_SIZES):
    """Per method: growth slopes over the measured sizes, fitted latency at the `at` sizes, and memory at the largest."""
    records = []
    for kind in SCALING_METHODS:
        mine = scaling[scaling["index"] == kind].dropna(subset=["latency_ms"]).sort_values("n_base")
        if len(mine) < 2:
            continue
        a, slope = power_law(mine["n_base"], mine["latency_ms"])
        record = dict(index=kind, latency_slope=slope,
                      distances_slope=power_law(mine["n_base"], mine["distances_per_query"])[1],
                      size_ratio=mine["n_base"].iloc[-1] / mine["n_base"].iloc[0],
                      latency_ratio=mine["latency_ms"].iloc[-1] / mine["latency_ms"].iloc[0])
        for n in at:
            record[f"latency_ms_at_{n}"] = a * n ** slope
            record[f"memory_gb_at_{n}"] = mine["bytes_per_vector"].iloc[-1] * n / 1e9
        records.append(record)
    return pd.DataFrame(records)


def part3_figure(scaling, fits):
    slopes = fits.set_index("index")
    panels = [("latency_ms", "latency_slope", "Search time at recall 0.95", "Milliseconds per query (1 thread)"),
              ("distances_per_query", "distances_slope", "Work per query at recall 0.95", "Distance computations per query")]
    sizes = sorted(scaling["n_base"].unique())
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.9), layout="constrained")
        placements = []
        for ax, (column, slope_column, title, ylabel) in zip(axes, panels):
            labels = []
            for kind in SCALING_METHODS:
                mine = scaling[scaling["index"] == kind].dropna(subset=[column]).sort_values("n_base")
                if mine.empty:
                    continue
                style = INDEX_STYLE[kind]
                ax.plot(mine["n_base"], mine[column], color=style["color"], marker=style["marker"], label=style["label"],
                        markeredgecolor="white", markeredgewidth=0.8)
                if kind in slopes.index:
                    end = (mine["n_base"].iloc[-1], mine[column].iloc[-1])
                    labels.append((f"slope {slopes.loc[kind, slope_column]:.2f}", end, (6, 0), "left", "center", "data"))
            ax.set(xscale="log", yscale="log", title=title, xlabel="Base vectors (N)", ylabel=ylabel)
            ax.set_xlim(sizes[0] / 1.5, sizes[-1] * 3.5)
            ax.set_xticks(sizes)
            ax.xaxis.set_major_formatter(ticker.FuncFormatter(compact))
            ax.xaxis.set_minor_formatter(ticker.NullFormatter())
            log_ticks(ax.yaxis)
            placements.append(labels)
        handles, names = axes[0].get_legend_handles_labels()
        fig.legend(handles, names, loc="outside upper center", ncols=len(names))
        fig.canvas.draw()
        for ax, labels in zip(axes, placements):
            direct_labels(ax, labels)
    return fig


def part3_assets(results):
    """Draws the Part 3 figure and writes its tables; returns the figure and the report versions of the tables."""
    scaling = scaling_table(results)
    fits = scaling_fit_table(scaling)
    fig = part3_figure(scaling, fits)
    save_figure(fig, "part3_scaling")
    q4 = {}
    for n in Q4_SIZES:
        q4[f"latency_ms_at_{n}"] = (f"ms/query at {compact(n)} (fit)", "{:.2f}".format)
        q4[f"memory_gb_at_{n}"] = (f"GB at {compact(n)}", "{:.2f}".format)
    tables = {
        "part3_scaling": pretty_table(scaling, {
            "n_base": ("N", "{:,}".format), "index": ("Method", NAME["index"]), "setting": ("Setting for 0.95", str),
            "recall_at_10": ("Recall", RECALL), "qps": ("QPS", format_qps), "latency_ms": ("ms/query", "{:.3f}".format),
            "distances_per_query": ("Distances/query", "{:,.0f}".format), "build_time_s": ("Build (s)", "{:.1f}".format),
            "bytes_per_vector": ("Bytes/vector", "{:.0f}".format)}),
        "part3_growth": pretty_table(fits, {
            "index": ("Method", NAME["index"]), "latency_slope": ("Latency slope", "{:.2f}".format),
            "distances_slope": ("Distance slope", "{:.2f}".format), "size_ratio": ("N grows", "{:.0f}x".format),
            "latency_ratio": ("Latency grows", "{:.1f}x".format), **q4}),
    }
    for name, table in tables.items():
        write_table(table, name)
    return fig, tables

In [ ]:
%%ipytest -qq

def toy_scaling_results():
    """Exact cost ~ N, IVF-Flat ~ sqrt(N), HNSW ~ log N, at three sizes; every index reaches 0.95 at its 2nd setting."""
    rows = []
    for n in (1_000, 4_000, 16_000):
        common = dict(part=3, section=str(n), dataset="sift", variant="", n_base=n, build_time_s=1.0,
                      index_bytes=n * 600, bytes_per_vector=600.0)
        rows.append(dict(common, index="exact", build_params="{}", param_name="", param_value=0, split="report",
                         recall_at_10=1.0, qps=1e6 / n, distances_per_query=float(n)))
        for kind, params, name, base_cost in (("ivf_flat", {"nlist": scaling_nlist(n)}, "nprobe", np.sqrt(n)),
                                              ("hnsw", {"M": 16, "efC": 200}, "efSearch", np.log(n))):
            for value, recall in ((8, 0.90), (16, 0.96), (32, 0.99)):
                cost = base_cost * value
                for split in ("tune", "report"):
                    rows.append(dict(common, index=kind, build_params=json.dumps(params, sort_keys=True),
                                     param_name=name, param_value=value, split=split, recall_at_10=recall,
                                     qps=1e6 / cost, distances_per_query=cost))
    return pd.DataFrame(rows)


def test_scaling_nlist_follows_the_square_root():
    assert [scaling_nlist(n) for n in (100_000, 300_000, 1_000_000)] == [1265, 2191, 4000]


def test_scaling_table_takes_the_operating_point_at_each_size():
    table = scaling_table(toy_scaling_results())
    assert len(table) == 9
    ivf = table[table["index"] == "ivf_flat"]
    assert (ivf["setting"].str.endswith("nprobe=16")).all()
    assert ivf["latency_ms"].tolist() == pytest.approx([1000 * 16 * np.sqrt(n) / 1e6 for n in (1_000, 4_000, 16_000)])


def test_growth_slopes_recover_n_sqrt_n_and_log_n():
    fits = scaling_fit_table(scaling_table(toy_scaling_results())).set_index("index")
    assert fits.loc["exact", "latency_slope"] == pytest.approx(1.0)
    assert fits.loc["ivf_flat", "latency_slope"] == pytest.approx(0.5)
    assert 0 < fits.loc["hnsw", "latency_slope"] < 0.3
    assert fits.loc["exact", "latency_ms_at_10000000"] == pytest.approx(10_000)  # 1 ms per 1,000 vectors
    assert fits.loc["exact", "memory_gb_at_10000000"] == pytest.approx(6.0)


def test_power_law_fit_is_exact_on_a_power_law():
    assert power_law([10, 100, 1000], [3, 30, 300]) == pytest.approx((0.3, 1.0))


def test_scaling_figure_has_both_panels_and_every_method(tmp_path):
    scaling = scaling_table(toy_scaling_results())
    fig = part3_figure(scaling, scaling_fit_table(scaling))
    for ax in fig.axes:
        assert sorted(l.get_label() for l in ax.get_lines()) == ["Exact search", "HNSW", "IVF-Flat"]
    plt.close(fig)

### Part 4: what noise does to each index
- **Relative contrast** at each noise level σ: the median, over report-half queries, of mean distance ÷ nearest-neighbour distance. It is stored in every Part 4 row.
- **Recall at fixed effort:** each index's 0.95 setting chosen on the control's (σ = 0) tune half, applied at every σ.
- **Cost of getting back to 0.95:** each σ's own operating point (chosen on its tune half), with its QPS and distance computations. Exact search is the control for the cost: N and d don't change with σ.

In [ ]:
NOISE_INDEXES = ["ivf_flat", "hnsw"]


def noise_table(results, target=0.95):
    """Per noise level and method: relative contrast, the operating point at `target`, and the report-half recall at the
    setting that reached `target` on the control's tune half."""
    part4 = results[results["part"] == 4]
    control = part4[part4["sigma"] == 0]
    reference = {kind: pick_operating_point(control[control["index"] == kind], target) for kind in NOISE_INDEXES}
    records = []
    for sigma in sorted(part4["sigma"].unique()):
        rows = part4[part4["sigma"] == sigma]
        for kind in ["exact", *NOISE_INDEXES]:
            mine = rows[rows["index"] == kind]
            if mine.empty:
                continue
            record = dict(sigma=sigma, relative_contrast=rows["relative_contrast"].iloc[0], index=kind)
            point = mine.iloc[0] if kind == "exact" else pick_operating_point(mine, target)
            if point is None:
                record["setting"] = "not reached"
            else:
                record.update(setting=describe_setting(point), recall_at_10=point["recall_at_10"], qps=point["qps"],
                              distances_per_query=point["distances_per_query"])
            if reference.get(kind) is not None:
                same = mine[(mine["split"] == "report") & (mine["param_value"] == reference[kind]["param_value"])]
                record["control_setting"] = describe_setting(reference[kind])
                record["recall_at_control_setting"] = same["recall_at_10"].iloc[0] if len(same) else np.nan
            records.append(record)
    return pd.DataFrame(records)


def part4_figure(table):
    sigmas = sorted(table["sigma"].unique())
    position = {sigma: i for i, sigma in enumerate(sigmas)}
    at = lambda frame: [position[s] for s in frame["sigma"]]
    marks = dict(markeredgecolor="white", markeredgewidth=0.8)
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, 3, figsize=(6.8, 2.7), layout="constrained")
        contrast = table.drop_duplicates("sigma")
        axes[0].plot(at(contrast), contrast["relative_contrast"], color=INK["secondary"], marker="o", **marks)
        axes[0].set(title="Relative contrast", ylabel="Mean ÷ nearest distance (median)")
        for kind in NOISE_INDEXES:
            mine = table[table["index"] == kind]
            style = INDEX_STYLE[kind]
            axes[1].plot(at(mine), mine["recall_at_control_setting"], color=style["color"], marker=style["marker"], **marks)
        axes[1].axhline(0.95, color=INK["axis"], linewidth=0.6, zorder=1)
        axes[1].set(title="Recall at fixed setting", ylabel="Recall@10 (control's 0.95 setting)")
        missing = []
        for kind in ["exact", *NOISE_INDEXES]:
            mine = table[table["index"] == kind]
            style = INDEX_STYLE[kind]
            reached = mine.dropna(subset=["qps"])
            axes[2].plot(at(reached), reached["qps"], color=style["color"], marker=style["marker"], label=style["label"],
                         **marks)
            missing += [(f"{style['label']}: not reached", (position[s], 0.03), (0, 0), "right", "bottom",
                         ("data", "axes fraction")) for s in mine.loc[mine["qps"].isna(), "sigma"]]
        axes[2].set(yscale="log", title="Speed at recall 0.95", ylabel="Queries per second (1 thread)")
        log_ticks(axes[2].yaxis)
        for ax in axes:
            ax.set_xticks(range(len(sigmas)), ["0\n(control)" if s == 0 else f"{s:g}" for s in sigmas])
            ax.set_xlim(-0.3, len(sigmas) - 0.7)
            ax.set_xlabel("Noise σ (× data spread)")
        handles, names = axes[2].get_legend_handles_labels()
        fig.legend(handles, names, loc="outside upper center", ncols=len(names))
        fig.canvas.draw()
        direct_labels(axes[2], missing)
    return fig


def part4_assets(results):
    """Draws the Part 4 figure and writes its table; returns the figure and the report version of the table."""
    table = noise_table(results)
    fig = part4_figure(table)
    save_figure(fig, "part4_noise")
    tables = {"part4_noise": pretty_table(table, {
        "sigma": ("σ", "{:g}".format), "relative_contrast": ("Rel. contrast", "{:.3f}".format),
        "index": ("Method", NAME["index"]), "setting": ("Setting for 0.95", str), "recall_at_10": ("Recall", RECALL),
        "qps": ("QPS", format_qps), "distances_per_query": ("Distances/query", "{:,.0f}".format),
        "recall_at_control_setting": ("Recall at control's setting", RECALL)})}
    for name, frame in tables.items():
        write_table(frame, name)
    return fig, tables

In [ ]:
%%ipytest -qq

def toy_noise_results():
    """Control and one noisy level: IVF-Flat reaches 0.95 later when noisy, HNSW never does."""
    rows = []
    for sigma, contrast in ((0.0, 1.8), (0.5, 1.2)):
        common = dict(part=4, section=f"sigma={sigma:g}", dataset="sift", variant=f"sigma={sigma:g}", n_base=1000,
                      build_time_s=1.0, index_bytes=600_000, bytes_per_vector=600.0, sigma=sigma,
                      relative_contrast=contrast)
        rows.append(dict(common, index="exact", build_params="{}", param_name="", param_value=0, split="report",
                         recall_at_10=1.0, qps=100.0, distances_per_query=1000.0))
        sweeps = {("ivf_flat", '{"nlist": 32}', "nprobe"): {2: 0.96 - sigma, 8: 0.99 - sigma / 4, 32: 1.0},
                  ("hnsw", '{"M": 16, "efC": 200}', "efSearch"): {16: 0.97 - sigma, 64: 0.99 - sigma / 2}}
        for (kind, params, name), measured in sweeps.items():
            for value, recall in measured.items():
                for split in ("tune", "report"):
                    rows.append(dict(common, index=kind, build_params=params, param_name=name, param_value=value,
                                     split=split, recall_at_10=recall, qps=10_000 / value, distances_per_query=50.0 * value))
    return pd.DataFrame(rows)


def test_noise_table_measures_both_kinds_of_breaking():
    table = noise_table(toy_noise_results()).set_index(["sigma", "index"])
    assert table.loc[(0.0, "ivf_flat"), "setting"] == "nlist=32, nprobe=2"
    assert table.loc[(0.5, "ivf_flat"), "setting"] == "nlist=32, nprobe=32"            # more lists to get back to 0.95
    assert table.loc[(0.5, "ivf_flat"), "recall_at_control_setting"] == pytest.approx(0.46)  # nprobe=2 when noisy
    assert table.loc[(0.5, "hnsw"), "setting"] == "not reached"
    assert table.loc[(0.5, "exact"), "qps"] == 100.0 and table.loc[(0.5, "exact"), "relative_contrast"] == 1.2


def test_noise_figure_has_three_panels_and_marks_what_was_not_reached():
    fig = part4_figure(noise_table(toy_noise_results()))
    assert len(fig.axes) == 3
    assert "HNSW: not reached" in [text.get_text() for text in fig.axes[2].texts]
    plt.close(fig)

### Part 1 tables, the machine, and how steady the timings were
- **Part 1:** the agreement check against the stored neighbours, and every exact-search timing with its speed-up over one query at a time.
- **Machine:** every distinct value of each machine field across `results.csv`. One value per field means one machine ran everything.
- **Run quality** (Q5): the spread between the timed passes of every one-at-a-time setting, (slowest − fastest) / fastest; the load average while rows were measured; and exact search on SIFT measured twice at the same N, in Parts 1 and 3.

`report_assets` rebuilds every figure and table from `results.csv`; `05_report_assets.ipynb` calls it.

In [ ]:
METHOD_LABEL = {"one_at_a_time": "one query at a time", "batch": "one batch"}
MACHINE_FIELDS = {
    "cpu_model": "CPU", "logical_cpus": "Logical CPUs", "ram_gb": "RAM (GiB)", "os": "OS",
    "governor": "Frequency governor", "timing_cpu": "Timing CPU (pinned)", "build_threads": "Build threads",
    "python": "Python", "numpy": "numpy", "faiss": "faiss-cpu", "h5py": "h5py", "pandas": "pandas",
    "matplotlib": "matplotlib", "blas": "BLAS libraries",
}


def part1_agreement_table(results):
    rows = results[(results["part"] == 1) & (results["method"] == "agreement")]
    return rows[["dataset", "n_queries", "recall_at_10", "recall_ours_vs_stored", "queries_below_full_recall",
                 "identical_top10_share", "max_abs_distance_diff", "zero_norm_base_vectors"]].reset_index(drop=True)


def part1_timing_table(results):
    """Every Part 1 timing, with its speed-up over one query at a time on the same dataset."""
    rows = results[(results["part"] == 1) & (results["method"] != "agreement")]
    table = rows[["dataset", "variant", "dim", "n_base", "method", "search_threads", "n_queries", "qps"]]
    table = table.reset_index(drop=True)
    single = table[table["method"] == "one_at_a_time"].set_index("dataset")["qps"]
    return table.assign(ms_per_query=1000 / table["qps"], speed_up=table["qps"] / table["dataset"].map(single))


def part1_assets(results):
    tables = {
        "part1_agreement": pretty_table(part1_agreement_table(results), {
            "dataset": ("Dataset", NAME["dataset"]), "n_queries": ("Queries checked", "{:,}".format),
            "recall_at_10": ("Recall@10, stored vs ours", RECALL), "recall_ours_vs_stored": ("Recall@10, ours vs stored", RECALL),
            "identical_top10_share": ("Identical top-10 sets", "{:.1%}".format),
            "max_abs_distance_diff": ("Largest distance difference", "{:.1e}".format)}),
        "part1_timing": pretty_table(part1_timing_table(results), {
            "dataset": ("Dataset", NAME["dataset"]), "n_base": ("Vectors", "{:,}".format), "dim": ("d", str),
            "variant": ("Dimension check", str), "method": ("Method", METHOD_LABEL.get),
            "search_threads": ("Threads", str), "qps": ("QPS", format_qps), "ms_per_query": ("ms/query", "{:.3f}".format),
            "speed_up": ("Speed-up", "{:.1f}x".format)}),
    }
    for name, table in tables.items():
        write_table(table, name)
    return tables


def machine_table(results):
    """Every distinct value of each machine field across `results` (one value each when one machine ran everything)."""
    return pd.DataFrame([dict(Field=label, Value="; ".join(str(value) for value in results[column].dropna().unique()))
                         for column, label in MACHINE_FIELDS.items() if column in results])


def repeat_spread(times_s):
    """(slowest - fastest) / fastest of one setting's timed passes."""
    times = json.loads(times_s)
    return (max(times) - min(times)) / min(times)


def run_quality_table(results):
    timed = results[results["times_s"].notna() & ~results["method"].isin(["batch", "agreement"])]
    spread = timed["times_s"].map(repeat_spread)
    records = [("One-at-a-time settings timed", f"{len(timed):,}"),
               ("Median spread between timed passes", f"{spread.median():.1%}"),
               ("90th percentile of that spread", f"{spread.quantile(0.9):.1%}"),
               ("Settings whose passes differ by more than 10%", f"{(spread > 0.10).mean():.1%}")]
    if "load_avg_1m" in results and results["load_avg_1m"].notna().any():
        load = results["load_avg_1m"].dropna()
        records.append(("1-minute load average while measuring (median, max)", f"{load.median():.2f}, {load.max():.2f}"))
    part1 = results[(results["part"] == 1) & (results["dataset"] == "sift") & (results["method"] == "one_at_a_time")]
    part3 = results[results["part"] == 3]
    if len(part1) and len(part3):
        same = part3[(part3["index"] == "exact") & (part3["n_base"] == part1["n_base"].iloc[0])]
        if len(same):
            records.append((f"Exact-search QPS on SIFT at N = {int(part1['n_base'].iloc[0]):,}, Part 3 / Part 1",
                            f"{same['qps'].iloc[0] / part1['qps'].iloc[0]:.3f}"))
    return pd.DataFrame(records, columns=["Measure", "Value"])


def report_assets(results):
    """Every figure and table in the report, from `results` alone; returns the figures and the report tables by name."""
    figures, tables = {}, dict(part1_assets(results))
    for name, assets in (("part2_recall_qps", part2_assets), ("part3_scaling", part3_assets), ("part4_noise", part4_assets)):
        figures[name], more = assets(results)
        tables.update(more)
    for name, table in (("machine", machine_table(results)), ("run_quality", run_quality_table(results))):
        tables[name] = write_table(table, name)
    return figures, tables

In [ ]:
%%ipytest -qq

def toy_part1_results():
    common = dict(part=1, dataset="sift", section="sift", variant="", dim=128, n_base=1000, search_threads=1,
                  n_queries=100)
    return pd.DataFrame([
        dict(common, method="agreement", recall_at_10=1.0, recall_ours_vs_stored=1.0, queries_below_full_recall=0,
             identical_top10_share=0.99, max_abs_distance_diff=1e-5, zero_norm_base_vectors=0, qps=np.nan),
        dict(common, method="one_at_a_time", qps=500.0, times_s="[0.20, 0.22, 0.21]"),
        dict(common, method="batch", qps=1500.0, times_s="[0.07, 0.07, 0.07]"),
        dict(common, method="batch", search_threads=12, qps=8000.0, times_s="[0.01, 0.01, 0.01]"),
    ])


def test_part1_timing_speed_up_is_relative_to_one_query_at_a_time():
    table = part1_timing_table(toy_part1_results())
    assert table["speed_up"].tolist() == pytest.approx([1.0, 3.0, 16.0])
    assert part1_agreement_table(toy_part1_results())["recall_at_10"].tolist() == [1.0]


def test_machine_table_shows_every_distinct_value():
    one = pd.DataFrame(dict(cpu_model=["A", "A"], faiss=["1.0", "1.0"]))
    assert machine_table(one)["Value"].tolist() == ["A", "1.0"]
    two = pd.DataFrame(dict(cpu_model=["A", "B"]))
    assert machine_table(two)["Value"].tolist() == ["A; B"]


def test_repeat_spread():
    assert repeat_spread("[1.0, 1.2, 1.1]") == pytest.approx(0.2)


def test_run_quality_counts_only_one_at_a_time_settings():
    table = run_quality_table(toy_part1_results().assign(load_avg_1m=0.5)).set_index("Measure")["Value"]
    assert table["One-at-a-time settings timed"] == "1"
    assert table["Median spread between timed passes"] == "10.0%"

## Hypothesis guard
The Part 3 and Part 4 notebooks call `require_committed_hypothesis` first. It reads `hypothesis.md` as committed at `HEAD`, not the working copy, and refuses to run if that part's section is missing or still contains `_TODO_`.

In [ ]:
def hypothesis_section(text, part):
    """Body of the '## <part>...' section of hypothesis.md ('' if there is none)."""
    body, inside = [], False
    for line in text.splitlines():
        if line.startswith("## "):
            inside = line[3:].startswith(part)
        elif inside:
            body.append(line)
    return "\n".join(body).strip()


def hypothesis_ready(text, part):
    body = hypothesis_section(text, part)
    return bool(body) and "_TODO_" not in body


def require_committed_hypothesis(part):
    git = ["git", "-C", str(ROOT)]
    if subprocess.run(git + ["rev-parse", "--is-inside-work-tree"], capture_output=True).returncode != 0:
        print(f"warning: {ROOT} is not a git repository, so the {part} hypothesis commit cannot be checked")
        return
    committed = subprocess.run(git + ["show", "HEAD:hypothesis.md"], capture_output=True, text=True)
    if committed.returncode != 0 or not hypothesis_ready(committed.stdout, part):
        raise RuntimeError(f"The {part} hypothesis is not committed. Write it in hypothesis.md and commit it before running this experiment.")
    commit = subprocess.run(git + ["log", "-1", "--date=iso", "--format=%h %ad", "--", "hypothesis.md"],
                            capture_output=True, text=True).stdout.strip()
    print(f"{part} hypothesis committed in {commit}")

In [ ]:
%%ipytest -qq

SAMPLE = """# Hypotheses

## Part 3: Scaling
Exact search slows down linearly.

## Part 4: Hard geometry
_TODO_
"""


def test_sections_are_found():
    assert hypothesis_section(SAMPLE, "Part 3") == "Exact search slows down linearly."
    assert hypothesis_section(SAMPLE, "Part 4") == "_TODO_"
    assert hypothesis_section(SAMPLE, "Part 5") == ""


def test_only_filled_in_sections_are_ready():
    assert hypothesis_ready(SAMPLE, "Part 3")
    assert not hypothesis_ready(SAMPLE, "Part 4") and not hypothesis_ready(SAMPLE, "Part 5")

In [ ]:
print(f"mode={MODE}  results -> {RESULTS_DIR.relative_to(ROOT)}  timing CPU {TIMING_CPU}  build threads {BUILD_THREADS}")